In [16]:
# set environment
import pandas as pd

# paths
INPUT_PATH = "../00_data/raw/population.csv.gz"
OUTPUT_PATH = "../00_data/clean/state_population_clean.csv"

### Data Preparation

In [17]:
# load the data
df = pd.read_csv(INPUT_PATH)

# basic info
print(f"Raw data shape: {df.shape}")

# quick look at the raw data
df.head()

Raw data shape: (5918603, 8)


,YEAR,HHWT,CLUSTER,STATEFIP,PERNUM,CBPERNUM,PERWT,SLWT
0,2006,338.0,2006002062991,8,1,1,336.0,336.0
1,2006,338.0,2006002062991,8,2,2,208.0,208.0
2,2006,338.0,2006002062991,8,3,3,205.0,205.0
3,2006,83.0,2006002063001,8,1,1,82.0,82.0
4,2006,83.0,2006002063001,8,2,2,86.0,86.0


In [18]:
# choose the data we need to use
df = df[["YEAR", "STATEFIP", "PERWT"]].copy()

# ensure PERWT is numeric
df["PERWT"] = pd.to_numeric(df["PERWT"], errors="coerce")

# check missing values -> we don't to deal with missing values
df.isna().sum()

YEAR        0
STATEFIP    0
PERWT       0
dtype: int64

### Data Cleaning

In [19]:
# choose the data we need to use
df = df[["YEAR", "STATEFIP", "PERWT"]]

# catgorize groups for future use
florida_group = [12]  # FL
florida_controls = [13, 45, 37]  # GA, SC, NC
washington_group = [53]  # WA
washington_controls = [41, 30, 8]  # OR, MT, CO

# select states
selected_fips = (
    florida_group + florida_controls + washington_group + washington_controls
)
df = df[df["STATEFIP"].isin(selected_fips)]

# check all data again
print("Data shape after filtering selected states:", df.shape)
df.head()

Data shape after filtering selected states: (5918603, 3)


,YEAR,STATEFIP,PERWT
0,2006,8,336.0
1,2006,8,208.0
2,2006,8,205.0
3,2006,8,82.0
4,2006,8,86.0


## Basic data analysis

In [ ]:
# basic analysis -> check whether the sample is resonable
print(df["STATEFIP"].value_counts())
print(df.groupby(["STATEFIP", "YEAR"]).size())

STATEFIP
12    1889687
13     953168
37     949756
53     678179
8      505001
45     463328
41     383538
30      95946
Name: count, dtype: int64
STATEFIP  YEAR
8         2006    48020
          2007    48256
          2008    48807
          2009    49434
          2010    49731
                  ...  
53        2011    69762
          2012    69301
          2013    69593
          2014    70600
          2015    71804
Length: 80, dtype: int64


In [21]:
# group the data to (state, year)
state_year = df.groupby(["STATEFIP", "YEAR"]).agg(POP=("PERWT", "sum")).reset_index()

# check the data
print(f"State_year shape: {state_year.shape}")
state_year.head()

State_year shape: (80, 3)


,STATEFIP,YEAR,POP
0,8,2006,4753377.0
1,8,2007,4861515.0
2,8,2008,4939456.0
3,8,2009,5024748.0
4,8,2010,5049071.0


In [ ]:
# average population by state across all years -> to check if data imbalance will need to be addressed later
avg_pop_by_state = (
    state_year.groupby("STATEFIP")["POP"]
    .mean()
    .reset_index()
    .rename(columns={"POP": "AVG_POP"})
    .sort_values("AVG_POP", ascending=False)
)

avg_pop_by_state

,STATEFIP,AVG_POP
1,12,19014330.6
2,13,9817575.8
4,37,9532569.3
7,53,6775247.4
0,8,5101335.2
6,45,4631273.2
5,41,3860338.0
3,30,991085.3


In [23]:
# output file for future use
state_year.to_csv(OUTPUT_PATH, index=False)
print(f"Cleaned data saved to: {OUTPUT_PATH}")

Cleaned data saved to: ../00_data/clean/state_population_clean.csv
